In [ ]:
import pandas as pd
from IPython.display import display

from db.connection import get_session
from evaluation.recall import evaluate_recall, load_ground_truth, dataset_stats
from evaluation.mlflow_log import log_eval_to_mlflow
from similarity.weights import SimilarityWeights

# --- Config: change these and re-run to compare ---
WEIGHTS = SimilarityWeights(morphological=0.60, ecological=0.25, taxonomic=0.15)
TOP_K = 5

In [ ]:
session = get_session()
pairs = load_ground_truth(session)
print(f"{len(pairs)} ground truth pairs loaded ({len(pairs) * 2} bidirectional evaluations)")

result = evaluate_recall(session, pairs, weights=WEIGHTS, top_k=TOP_K)
print(f"Evaluation complete in {result.duration_s}s")

In [ ]:
# Recall@K summary
for k in [1, 3, 5]:
    print(f"Recall@{k}: {result.hits_at(k)}/{result.total} = {result.recall_at(k):.1%}")

if result.errors:
    print(f"\nErrors (species not found / missing embeddings): {result.errors}")

In [ ]:
# Log to MLflow
stats = dataset_stats(session)
ok = log_eval_to_mlflow(result, run_name=None, dataset_params=stats)
if ok:
    print("Logged to MLflow (experiment: eval_retrieval). View at http://localhost:5000")
else:
    print("MLflow logging skipped (service unavailable).")

In [ ]:
# Hits — successful retrievals sorted by rank
hits = [r for r in result.pair_results if r.rank is not None]
df_hits = pd.DataFrame([vars(r) for r in hits]).sort_values("rank")
display(df_hits[["query", "target", "rank", "sim_overall", "sim_morph", "sim_eco", "sim_taxon"]].round(3))

In [ ]:
# Misses — target not found in top-K
misses = [r for r in result.pair_results if r.rank is None]
df_misses = pd.DataFrame([vars(r) for r in misses])
display(df_misses[["query", "target", "error"]])